## Predicting congestion level

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

    - Importing data

In [2]:
data = pd.read_csv('../../data/traffic_data_cleaned_&_engineered.csv')

    - Defining Congestion Levels (Target)

In [3]:
low_threshold = data['traffic_volume'].quantile(0.33)
high_threshold = data['traffic_volume'].quantile(0.66)

    - Function for creating column

In [4]:
def map_congestion(vol):
    if vol <= low_threshold: return 0 # Low
    elif vol <= high_threshold: return 1 # Medium
    else: return 2 # High

In [5]:
data['congestion_level'] = data['traffic_volume'].apply(map_congestion)

    - Dividing Dependent and Independent features

In [6]:
X = data.drop(columns=['traffic_volume', 'congestion_level', 'weather_main'])
y = data['congestion_level']

In [7]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,traffic_volume,is_holiday,day_of_week,hour_sin,hour_cos,month_sin,month_cos,congestion_level
0,288.28,0.0,0.0,40,Clouds,5545,0,1,7.071068e-01,-0.707107,-0.866025,0.5,2
1,289.36,0.0,0.0,75,Clouds,4516,0,1,5.000000e-01,-0.866025,-0.866025,0.5,1
2,289.58,0.0,0.0,90,Clouds,4767,0,1,2.588190e-01,-0.965926,-0.866025,0.5,2
3,290.13,0.0,0.0,90,Clouds,5026,0,1,1.224647e-16,-1.000000,-0.866025,0.5,2
4,291.14,0.0,0.0,75,Clouds,4918,0,1,-2.588190e-01,-0.965926,-0.866025,0.5,2


    - Train Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    - Defining pipelines

In [9]:
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', XGBClassifier(random_state=42))
])

    - Comparing both model

In [10]:
for name, pipe in [("Random Forest", rf_pipeline), ("XGBoost", xgb_pipeline)]:
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"{name} Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Random Forest Accuracy: 0.9125
XGBoost Accuracy: 0.9131


    - XGBoost performing better so we hyperparameter tuning this model

In [11]:
# Hyperparameter Grid for XGBoost
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [3, 5, 7],
    'clf__learning_rate': [0.01, 0.1, 0.2]
}

grid_search = GridSearchCV(xgb_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Train Accuracy: {grid_search.best_score_:.4f}")

# Evaluate the Best Model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nFinal Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))


Best Parameters: {'clf__learning_rate': 0.2, 'clf__max_depth': 7, 'clf__n_estimators': 200}
Best Train Accuracy: 0.9117

Final Classification Report:
              precision    recall  f1-score   support

         Low       0.96      0.96      0.96      3083
      Medium       0.89      0.86      0.88      3250
        High       0.89      0.93      0.91      3259

    accuracy                           0.92      9592
   macro avg       0.92      0.92      0.92      9592
weighted avg       0.92      0.92      0.92      9592



In [13]:
print(f"{name} Accuracy with best params of xgboost: {accuracy_score(y_test, y_pred):.4f}")

XGBoost Accuracy with best params of xgboost: 0.9157


    - Saving model

In [12]:
model_dir = '../../models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

joblib.dump(best_model, os.path.join(model_dir, 'congestion_classification_pipeline.joblib'))

print(f"Tuned Congestion Model saved successfully in {model_dir}!")


Tuned Congestion Model saved successfully in ../../models!
